In [ ]:
from google.colab import files
uploaded = files.upload()

Saving heart (2).csv to heart (2).csv


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
import numpy as np

# Load the dataset
df = pd.read_csv('/content/heart (2).csv')

# Display the first 5 rows of the dataset
print("First 5 rows of the dataset:")
display(df.head())

# Display general information about the dataset
print("\nInformation about the dataset:")
df.info()

First 5 rows of the dataset:


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0



Information about the dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


In [ ]:
# Separate features (X) and target (y)
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include='object').columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns

print(f"Categorical columns: {list(categorical_cols)}")
print(f"Numerical columns: {list(numerical_cols)}")

# Create a column transformer for preprocessing
# One-hot encode categorical features and scale numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# Apply preprocessing
X_processed = preprocessor.fit_transform(X)

# Get feature names after one-hot encoding (optional, for better understanding)
# This part can be complex for a ColumnTransformer, but we'll proceed without explicit names for now.
# If needed, feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)

print("\nShape of processed features:", X_processed.shape)
print("Processed data snippet (first 5 rows, few columns):")
# Convert to DataFrame to view, as X_processed is a sparse matrix or numpy array
# This conversion is mainly for display purposes, actual models will work with the array.
display(pd.DataFrame(X_processed).head())

Categorical columns: ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
Numerical columns: ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']

Shape of processed features: (918, 20)
Processed data snippet (first 5 rows, few columns):


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,-1.433140,0.410909,0.825070,-0.551341,1.382928,-0.832432,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
1,-0.478484,1.491752,-0.171961,-0.551341,0.754157,0.105664,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
2,-1.751359,-0.129513,0.770188,-0.551341,-1.525138,-0.832432,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
3,-0.584556,0.302825,0.139040,-0.551341,-1.132156,0.574711,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
4,0.051881,0.951331,-0.034755,-0.551341,-0.581981,-0.832432,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0


In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (734, 20)
X_test shape: (184, 20)
y_train shape: (734,)
y_test shape: (184,)


In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Support Vector Machine': SVC(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

# Train and evaluate models
results = {}
print("\n--- Model Training and Evaluation (without PCA) ---")
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    print(f"{name} Accuracy: {accuracy:.4f}")

# Find the best model
best_model_name = max(results, key=results.get)
best_accuracy = results[best_model_name]

print(f"\nBest model (without PCA): {best_model_name} with accuracy: {best_accuracy:.4f}")


--- Model Training and Evaluation (without PCA) ---
Training Logistic Regression...
Logistic Regression Accuracy: 0.8533
Training Support Vector Machine...
Support Vector Machine Accuracy: 0.8696
Training Random Forest...
Random Forest Accuracy: 0.8804

Best model (without PCA): Random Forest with accuracy: 0.8804


In [ ]:
# Apply PCA
# Let's choose the number of components to retain, for example, 95% of variance or a fixed number.
# For demonstration, let's aim for a reasonable number of components, or a variance threshold.
# We can inspect explained variance ratio to make an informed decision.
pca = PCA(n_components=0.95) # Retain 95% of the variance
X_pca = pca.fit_transform(X_processed)

print(f"Original number of features: {X_processed.shape[1]}")
print(f"Number of features after PCA (retaining 95% variance): {X_pca.shape[1]}")
print(f"Explained variance ratio by each component:\n{pca.explained_variance_ratio_}")
print(f"Cumulative explained variance: {np.sum(pca.explained_variance_ratio_):.4f}")

# Split PCA-transformed data into training and testing sets
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(X_pca, y, test_size=0.2, random_state=42)

print(f"\nX_train_pca shape: {X_train_pca.shape}")
print(f"X_test_pca shape: {X_test_pca.shape}")

Original number of features: 20
Number of features after PCA (retaining 95% variance): 12
Explained variance ratio by each component:
[0.25137218 0.15334608 0.11336034 0.09797289 0.08000528 0.07223638
 0.04188982 0.0368846  0.03548957 0.03236191 0.02788408 0.02204725]
Cumulative explained variance: 0.9649

X_train_pca shape: (734, 12)
X_test_pca shape: (184, 12)


In [ ]:
# Initialize models again (to ensure fresh training)
models_pca = {
    'Logistic Regression (PCA)': LogisticRegression(random_state=42, max_iter=1000),
    'Support Vector Machine (PCA)': SVC(random_state=42),
    'Random Forest (PCA)': RandomForestClassifier(random_state=42)
}

# Train and evaluate models with PCA data
results_pca = {}
print("\n--- Model Training and Evaluation (with PCA) ---")
for name, model in models_pca.items():
    print(f"Training {name}...")
    model.fit(X_train_pca, y_train_pca)
    y_pred_pca = model.predict(X_test_pca)
    accuracy_pca = accuracy_score(y_test_pca, y_pred_pca)
    results_pca[name] = accuracy_pca
    print(f"{name} Accuracy: {accuracy_pca:.4f}")

# Find the best model with PCA
best_model_pca_name = max(results_pca, key=results_pca.get)
best_accuracy_pca = results_pca[best_model_pca_name]

print(f"\nBest model (with PCA): {best_model_pca_name} with accuracy: {best_accuracy_pca:.4f}")

print("\n--- Comparison of Model Accuracies ---")
print("Without PCA:")
for name, acc in results.items():
    print(f"  {name}: {acc:.4f}")
print("\nWith PCA:")
for name, acc_pca in results_pca.items():
    print(f"  {name}: {acc_pca:.4f}")

print(f"\nOverall Best Model (without PCA): {best_model_name} with accuracy {best_accuracy:.4f}")
print(f"Overall Best Model (with PCA): {best_model_pca_name} with accuracy {best_accuracy_pca:.4f}")


--- Model Training and Evaluation (with PCA) ---
Training Logistic Regression (PCA)...
Logistic Regression (PCA) Accuracy: 0.8533
Training Support Vector Machine (PCA)...
Support Vector Machine (PCA) Accuracy: 0.8587
Training Random Forest (PCA)...
Random Forest (PCA) Accuracy: 0.8424

Best model (with PCA): Support Vector Machine (PCA) with accuracy: 0.8587

--- Comparison of Model Accuracies ---
Without PCA:
  Logistic Regression: 0.8533
  Support Vector Machine: 0.8696
  Random Forest: 0.8804

With PCA:
  Logistic Regression (PCA): 0.8533
  Support Vector Machine (PCA): 0.8587
  Random Forest (PCA): 0.8424

Overall Best Model (without PCA): Random Forest with accuracy 0.8804
Overall Best Model (with PCA): Support Vector Machine (PCA) with accuracy 0.8587
